# Quantize and Register Ollama (Local)

This notebook is for **local machine** execution. It will:
1. Reuse or download HF checkpoint automatically
2. Convert checkpoint to F16 GGUF
3. Quantize GGUF (e.g. Q4_K_M)
4. Optionally register quantized GGUF into local Ollama


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
# If notebook is launched from a different folder, set manually:
# PROJECT_ROOT = Path('/Users/kerembozgan/Library/CloudStorage/GoogleDrive-bozgan.kerem4@gmail.com/My Drive/training-embedding')

MODEL_ID = 'ytu-ce-cosmos/Turkish-Gemma-9b-v0.1'
WORK_DIR = Path('/Users/kerembozgan/Desktop/models')
LLAMA_CPP_DIR = Path.home() / 'llama.cpp'
QUANT_TYPE = 'q4_k_m'
OLLAMA_MODEL_NAME = 'turkish-gemma-v01-q4km'

if not (PROJECT_ROOT / 'hf_to_gguf_local.py').exists():
    raise FileNotFoundError(f'hf_to_gguf_local.py not found under: {PROJECT_ROOT}')

WORK_DIR.mkdir(parents=True, exist_ok=True)

def load_hf_token(project_root: Path) -> str:
    token = os.environ.get('HF_TOKEN', '').strip()
    if token:
        return token
    env_path = project_root / '.env'
    if not env_path.exists():
        return ''
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        k = k.strip()
        if k.startswith('export '):
            k = k[len('export '):].strip()
        if k == 'HF_TOKEN':
            return v.strip().strip("\"").strip("'")
    return ''

HF_TOKEN = load_hf_token(PROJECT_ROOT)
print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'WORK_DIR={WORK_DIR}')
print(f'LLAMA_CPP_DIR={LLAMA_CPP_DIR}')
print('HF_TOKEN loaded from env/.env' if HF_TOKEN else 'HF_TOKEN not set; proceeding unauthenticated')


In [ ]:
import subprocess

# Install/refresh Python dependencies declared in pyproject.toml
subprocess.run(['uv', 'sync'], cwd=PROJECT_ROOT, check=True)


In [ ]:
import subprocess
import shlex

cmd = [
    'uv', 'run', 'python', str(PROJECT_ROOT / 'hf_to_gguf_local.py'),
    '--model-id', MODEL_ID,
    '--work-dir', str(WORK_DIR),
    '--llama-cpp-dir', str(LLAMA_CPP_DIR),
    '--quantization-type', QUANT_TYPE,
]

env = os.environ.copy()
if HF_TOKEN:
    env['HF_TOKEN'] = HF_TOKEN

print('Running:')
print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, env=env, check=True)


In [ ]:
model_name = MODEL_ID.split('/')[-1]
full_model_dir = WORK_DIR / f'{model_name}-full'
gguf_f16 = WORK_DIR / f'{model_name}.F16.gguf'
gguf_quant = WORK_DIR / f'{model_name}.{QUANT_TYPE.upper()}.gguf'

def describe_path(path: Path) -> str:
    if not path.exists():
        return f'{path} -> MISSING'
    if path.is_file():
        gib = path.stat().st_size / (1024**3)
        return f'{path} -> {gib:.2f} GiB'
    return f'{path} -> directory'

print(describe_path(full_model_dir))
print(describe_path(gguf_f16))
print(describe_path(gguf_quant))


In [ ]:
# Optional: register the quantized GGUF in LOCAL Ollama
import shutil
import subprocess
import tempfile

REGISTER_LOCAL_OLLAMA = False
NUM_CTX = 2048

if REGISTER_LOCAL_OLLAMA:
    ollama_bin = shutil.which('ollama')
    if not ollama_bin:
        raise RuntimeError('ollama binary not found in PATH')
    if not gguf_quant.exists():
        raise FileNotFoundError(f'Quantized GGUF not found: {gguf_quant}')

    with tempfile.NamedTemporaryFile('w', suffix='.modelfile', delete=False) as f:
        modelfile_path = Path(f.name)
        f.write(f'FROM {gguf_quant}\n')
        f.write(f'PARAMETER num_ctx {NUM_CTX}\n')

    try:
        subprocess.run([ollama_bin, 'create', OLLAMA_MODEL_NAME, '-f', str(modelfile_path)], check=True)
        subprocess.run([ollama_bin, 'list'], check=True)
    finally:
        modelfile_path.unlink(missing_ok=True)
else:
    print('Skipping local Ollama registration. Set REGISTER_LOCAL_OLLAMA=True to enable.')


In [ ]:
# Optional cleanup to save local disk AFTER successful quantization/registration
import shutil

CLEANUP_FULL_AND_F16 = False

if CLEANUP_FULL_AND_F16:
    shutil.rmtree(full_model_dir, ignore_errors=True)
    gguf_f16.unlink(missing_ok=True)
    print('Removed full model folder and F16 GGUF. Quantized GGUF kept.')
else:
    print('Cleanup skipped.')
